# NB_lab_test_trends

Builds gold table gold.lab_test_trends (one row per test type per calendar month) from `silver.lab_results`, enriched with month/quarter names from `silver.calendar`, including a month-over-month LAG trend on the abnormal rate, per the Silver-to-Gold STTM. Load type: full rebuild (CREATE OR REPLACE TABLE ... AS SELECT).

In [0]:
dbutils.widgets.text("catalog", "adb_ws_wecare", "Catalog")
catalog = dbutils.widgets.get("catalog")
silver, gold = f"{catalog}.silver", f"{catalog}.gold"
target = f"{gold}.lab_test_trends"
print(f"Target: {target}")

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold}")

spark.sql(f"""
CREATE OR REPLACE TABLE {target}
COMMENT 'Gold: one row per test type per calendar month with month-over-month abnormal-rate trend (STTM Silver to Gold)'
AS
WITH monthly AS (
  SELECT
    test_name,
    year(test_date)                                        AS year,
    month(test_date)                                       AS month,
    CAST(date_trunc('MONTH', MIN(test_date)) AS DATE)      AS month_start,
    COUNT(*)                                               AS total_tests,
    COUNT_IF(result_status = 'Abnormal')                   AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')                   AS critical_test_count
  FROM {silver}.lab_results
  GROUP BY test_name, year(test_date), month(test_date)
),
rates AS (
  SELECT
    *,
    CAST(abnormal_test_count / total_tests * 100 AS DECIMAL(5,2)) AS abnormal_rate_pct,
    CAST(critical_test_count / total_tests * 100 AS DECIMAL(5,2)) AS critical_rate_pct
  FROM monthly
),
trend AS (
  SELECT
    *,
    LAG(abnormal_rate_pct) OVER (PARTITION BY test_name ORDER BY year, month) AS prev_month_abnormal_rate_pct
  FROM rates
)
SELECT
  CAST(t.test_name AS STRING)                                   AS test_name,
  CAST(t.year AS INT)                                           AS year,
  CAST(t.month AS INT)                                          AS month,
  CAST(c.month_name AS STRING)                                  AS month_name,
  CAST(c.quarter_name AS STRING)                                AS quarter_name,
  CAST(t.total_tests AS INT)                                    AS total_tests,
  CAST(t.abnormal_test_count AS INT)                            AS abnormal_test_count,
  CAST(t.critical_test_count AS INT)                            AS critical_test_count,
  t.abnormal_rate_pct                                           AS abnormal_rate_pct,
  t.critical_rate_pct                                           AS critical_rate_pct,
  CAST(t.prev_month_abnormal_rate_pct AS DECIMAL(5,2))          AS prev_month_abnormal_rate_pct,
  CAST(t.abnormal_rate_pct - t.prev_month_abnormal_rate_pct AS DECIMAL(6,2)) AS abnormal_rate_change_pct
FROM trend t
LEFT JOIN {silver}.calendar c ON c.date_key = CAST(date_format(t.month_start, 'yyyyMMdd') AS INT)
""")
print(f"Rebuilt {target}")

In [0]:
updated_df = spark.table(target)
record_count = updated_df.count()
print(f"{target}: {record_count} records")
dbutils.notebook.exit(str(record_count))